In [1]:
# Run this cell first in Colab
!pip install --quiet kagglehub scikit-learn pandas flask-ngrok spacy
!python -m spacy download en_core_web_sm


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 43.9 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


In [2]:
import os, glob
import re
import pandas as pd
import numpy as np

import kagglehub

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report

import spacy
nlp = spacy.load("en_core_web_sm")

print("Downloading dataset...")
path = kagglehub.dataset_download("elvinagammed/chatbots-intent-recognition-dataset")
print("Dataset path:", path)

# List CSVs
csvs = glob.glob(os.path.join(path, "*.csv"))
print("CSV files found:", csvs)


100%|██████████| 16.9k/16.9k [00:00<00:00, 8.73MB/s]

Extracting files...
Dataset path: /root/.cache/kagglehub/datasets/elvinagammed/chatbots-intent-recognition-dataset/versions/1
CSV files found: []


In [ ]:
# Adjust filenames if dataset has different names. Common: intents.csv or similar.
# Try to find a CSV that looks like intent data (contains 'intent' or 'text' columns)
file_candidates = csvs
df = None
for c in file_candidates:
    try:
        tmp = pd.read_csv(c, encoding='utf-8', on_bad_lines='skip')
        if any(col.lower() in ['intent','label','text','utterance','sentence'] for col in tmp.columns):
            df = tmp
            print("Loaded", c)
            break
    except Exception as e:
        print("Could not read", c, e)

if df is None:
    raise RuntimeError("Couldn't auto-load an intent CSV. Inspect the dataset folder and pick the right CSV.")

print("Columns:", df.columns.tolist())
display(df.head(10))


In [ ]:
# Try to find the text and intent columns
text_col = None
intent_col = None
for c in df.columns:
    cl = c.lower()
    if any(x in cl for x in ['text','utter','sentence','message']):
        text_col = c
    if any(x in cl for x in ['intent','label','tag','class']):
        intent_col = c

if text_col is None or intent_col is None:
    print("Could not automatically detect text/intent columns. Columns:", df.columns.tolist())
    raise RuntimeError("Rename appropriate columns to be detected automatically.")

# Keep only those two columns and drop NAs
df = df[[text_col, intent_col]].dropna().rename(columns={text_col: 'text', intent_col: 'intent'})
df['text'] = df['text'].astype(str).str.strip()
df['intent'] = df['intent'].astype(str).str.strip()

print("Sample:")
display(df.sample(8))
print("Intent distribution:")
print(df['intent'].value_counts())


In [ ]:
# train-test split
train_df, test_df = train_test_split(df, test_size=0.2, random_state=42, stratify=df['intent'])

# pipeline: TF-IDF + Logistic Regression (fast & effective for intents)
pipeline = Pipeline([
    ('tfidf', TfidfVectorizer(ngram_range=(1,2), max_features=10000)),
    ('clf', LogisticRegression(max_iter=1000))
])

pipeline.fit(train_df['text'], train_df['intent'])
y_pred = pipeline.predict(test_df['text'])

print("Evaluation on test set:")
print("Accuracy:", accuracy_score(test_df['intent'], y_pred))
print("Precision:", precision_score(test_df['intent'], y_pred, average='weighted', zero_division=0))
print("Recall   :", recall_score(test_df['intent'], y_pred, average='weighted', zero_division=0))
print("F1-score :", f1_score(test_df['intent'], y_pred, average='weighted', zero_division=0))
print("\nClassification report:")
print(classification_report(test_df['intent'], y_pred, zero_division=0))


In [ ]:
# Basic entity rules for common customer service chatbot tasks (dates, times, numbers, product names heuristics)
def extract_entities(text):
    doc = nlp(text)
    ents = {}
    # Dates / times / numbers using spaCy entities
    for ent in doc.ents:
        if ent.label_ in ("DATE","TIME"):
            ents.setdefault("date_time", []).append(ent.text)
        if ent.label_ in ("CARDINAL","QUANTITY","MONEY","PERCENT"):
            ents.setdefault("number", []).append(ent.text)
        if ent.label_ in ("ORG","PRODUCT"):
            ents.setdefault("product", []).append(ent.text)
    # Simple regex for phone / order id
    phone = re.findall(r'\b(?:\+?\d{10,15})\b', text)
    if phone:
        ents['phone'] = phone
    orderid = re.findall(r'\b(order|id)[\s:#-]*([A-Za-z0-9\-]+)\b', text, flags=re.I)
    if orderid:
        ents['order_id'] = [m[1] for m in orderid]
    # fallback: extract quoted phrases as product names
    quotes = re.findall(r'["“](.+?)["”]', text)
    if quotes:
        ents.setdefault('product', []).extend(quotes)
    return ents

# quick test
print(extract_entities("I want to book a table for 2 on 12th Aug at 7pm. My phone +923001234567. Order id: ORD-345"))


In [ ]:
# Define simple response templates per intent
RESPONSE_TEMPLATES = {
    "greeting": ["Hello! How can I help you today?", "Hi there! What can I do for you?"],
    "goodbye": ["Goodbye! Have a great day.", "See you later — reach out if you need anything else."],
    "thanks": ["You're welcome!", "Anytime — happy to help!"],
    # service-specific
    "order_status": ["I can check that. Can you provide your order id?", "Please share your order id so I can look it up."],
    "book_table": ["Sure — for what date and time would you like the booking?", "I can help with booking. Which date & time?"],
    "book_table_confirm": ["Booking confirmed for {date_time} for {party_size} people.", "Your table is booked on {date_time} for {party_size} guests."],
    "fallback": ["Sorry, I didn't understand. Can you rephrase?", "I'm not sure I got that — can you tell me differently?"]
}

import random

# Dialog state holder (for demo; in production use per-user persistent state)
class DialogState:
    def __init__(self):
        self.slots = {}

    def set_slot(self, key, value):
        self.slots[key] = value

    def get_slot(self, key):
        return self.slots.get(key)

    def reset(self):
        self.slots = {}

state = DialogState()

def choose_response(intent, entities):
    # Examples of simple logic: booking flow
    if intent == 'book_table':
        # try to fill slots
        if 'date_time' in entities:
            state.set_slot('date_time', entities['date_time'][0])
        if 'number' in entities:
            # take first cardinal as party size
            state.set_slot('party_size', entities['number'][0])
        # if both slots available -> confirm
        if state.get_slot('date_time') and state.get_slot('party_size'):
            resp = random.choice(RESPONSE_TEMPLATES['book_table_confirm']).format(
                date_time=state.get_slot('date_time'),
                party_size=state.get_slot('party_size')
            )
            state.reset()
            return resp
        else:
            return random.choice(RESPONSE_TEMPLATES['book_table'])
    # order status
    if intent == 'order_status':
        if 'order_id' in entities:
            return f"I found order `{entities['order_id'][0]}` — it's currently in transit. Estimated delivery: 2 days."
        else:
            return random.choice(RESPONSE_TEMPLATES['order_status'])
    # greeting/thanks/goodbye
    if intent in RESPONSE_TEMPLATES:
        return random.choice(RESPONSE_TEMPLATES[intent])
    return random.choice(RESPONSE_TEMPLATES['fallback'])


In [ ]:
def nlu_predict(text):
    # intent prediction
    intent = pipeline.predict([text])[0]
    # extract entities
    entities = extract_entities(text)
    return intent, entities

def chat_once(text):
    intent, entities = nlu_predict(text)
    response = choose_response(intent, entities)
    return {"text": text, "intent": intent, "entities": entities, "response": response}

# quick interactive tests
examples = [
    "Hello there",
    "I want to book a table for 4 on September 5 at 8pm",
    "What's the status of my order ORD-3456?",
    "Thanks, bye!"
]

for ex in examples:
    print(chat_once(ex))


In [ ]:
# Evaluate intent classifier on test set (already computed earlier)
y_true = test_df['intent'].tolist()
y_pred_all = pipeline.predict(test_df['text'])
print("Intent classification report:")
print(classification_report(y_true, y_pred_all, zero_division=0))


In [ ]:
# Optional: Run this cell to start a small web endpoint (in local env). For Colab use flask-ngrok or pyngrok.
from flask import Flask, request, jsonify
from flask_ngrok import run_with_ngrok

app = Flask(__name__)
run_with_ngrok(app)  # starts ngrok when app.run() is called

@app.route("/chat", methods=["POST"])
def chat_endpoint():
    data = request.get_json(force=True)
    text = data.get("text", "")
    out = chat_once(text)
    return jsonify(out)

# To run the server (Uncomment below). In Colab it will provide a public ngrok URL.
# app.run()
